# TARDIS_MODEL

Ce projet 

## Importation des modules nécessaires

**hello**

In [1]:
import subprocess
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Chargement de la donnée 

In [2]:
# Vérification de l'existance du dataset clean
subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "tardis_eda.ipynb"], check=True)

# Lecture du dataset nettoyé
df = pd.read_csv("cleaned_dataset.csv", sep=";")

# Taille du dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du dataset
print("\nAperçu du dataset:")
df.head()

[NbConvertApp] Converting notebook tardis_eda.ipynb to notebook
[NbConvertApp] Writing 158159 bytes to tardis_eda.nbconvert.ipynb


Nombre d'échantillons: 11487
Nombre de caractéristiques: 25

Aperçu du dataset:


Date   Service   Departure station     Arrival station  \
0  2018-01  National    BORDEAUX ST JEAN  PARIS MONTPARNASSE   
1  2018-01  National             LE MANS  PARIS MONTPARNASSE   
2  2018-01  National  PARIS MONTPARNASSE   LA ROCHELLE VILLE   
3  2018-01  National  PARIS MONTPARNASSE              NANTES   
4  2018-01  National            POITIERS  PARIS MONTPARNASSE   

   Average journey time  Number of scheduled trains  \
0                141.00                       870.0   
1                 56.00                       406.0   
2                166.00                       226.0   
3                216.21                       508.0   
4                 94.00                       472.0   
   Average journey time  Number of scheduled trains  \
0                141.00                       870.0   
1                 56.00                       406.0   
2                166.00                       226.0   
3                216.21                       508.0   
4                 94.00                       472.0   

   Number of cancelled trains  Number of trains delayed at departure  \
0                         5.0                                  289.0   
1                         1.0                                  213.0   
2                         0.0                                   21.0   
3                         3.0                                   71.0   
4                         4.0                                  224.0   
   Number of cancelled trains  Number of trains delayed at departure  \
0                         5.0                                  289.0   
1                         1.0                                  213.0   
2                         0.0                                   21.0   
3                         3.0                                   71.0   
4                         4.0                                  224.0   

   Average delay of late trains at departure  \
0                                  11.247809   
1                                   8.479969   
2                                   6.239683   
3                                   7.235211   
4                                   6.784673   
   Average delay of late trains at departure  \
0                                  11.247809   
1                                   8.479969   
2                                   6.239683   
3                                   7.235211   
4                                   6.784673   

   Average delay of all trains at departure  ...  \
0                                  3.693179  ...   
1                                  4.567119  ...   
2                                  0.286283  ...   
3                                  0.980000  ...   
4                                  3.229701  ...   
   Average delay of all trains at departure  ...  \
0                                  3.693179  ...   
1                                  4.567119  ...   
2                                  0.286283  ...   
3                                  0.980000  ...   
4                                  3.229701  ...   

   Number of trains delayed > 30min  Number of trains delayed > 60min  \
0                              44.0                               8.0   
1                               9.0                               4.0   
2                               6.0                               1.0   
3                              18.0                               3.0   
4                              10.0                               0.0   

   Pct delay due to external causes  Pct delay due to infrastructure  \
0                         36.134454                        31.092437   
1                         20.000000                        35.000000   
2                         22.222222                        27.777778   
3                         33.333333                        22.222222   
4                         15.789474                        45.614035   

   Pct delay due to t

In [10]:
stations = list(df['Departure station'].unique())
for i in range(len(stations)):
    print(i, stations[i])

0 BORDEAUX ST JEAN
1 LE MANS
2 PARIS MONTPARNASSE
3 POITIERS
4 ST MALO
5 TOULOUSE MATABIAU
6 VANNES
7 REIMS
8 DOUAI
9 LILLE
10 PARIS NORD
11 AIX EN PROVENCE TGV
12 ANNECY
13 GRENOBLE
14 LYON PART DIEU
15 MACON LOCHE
16 MARSEILLE ST CHARLES
17 MONTPELLIER
18 PARIS LYON
19 BARCELONA
20 FRANCFORT
21 MADRID
22 ST PIERRE DES CORPS
23 TOURS
24 PARIS EST
25 PARIS VAUGIRARD
26 RENNES
27 BESANCON FRANCHE COMTE TGV
28 DIJON VILLE
29 PERPIGNAN
30 VALENCE ALIXAN TGV
31 STUTTGART
32 METZ
33 NANCY
34 NANTES
35 LE CREUSOT MONTCEAU MONTCHANIN
36 MARNE LA VALLEE
37 NICE VILLE
38 ITALIE
39 LAUSANNE
40 ANGOULEME
41 BREST
42 LA ROCHELLE VILLE
43 Paris Montparnasse
44 QUIMPER
45 STRASBOURG
46 AVIGNON TGV
47 MULHOUSE VILLE
48 SAINT ETIENNE CHATEAUCREUX
49 GENEVE
50 ANGERS SAINT LAUD
51 LAVAL
52 ARRAS
53 DUNKERQUE
54 TOURCOING
55 BELLEGARDE (AIN)
56 CHAMBERY CHALLES LES EAUX
57 NIMES
58 TOULON
59 ZURICH
60  ITALIE 
61 Paris Lyon
62 marseille st charles
63  PARIS LYON 
64 paris montparnasse
65 Laval
66 Nice V

In [9]:
len(stations)

131

In [3]:
# Informations sur le dataset
print("Informations sur le dataset:")
df.info()

Informations sur le dataset:
<class 'pandas.DataFrame'>
RangeIndex: 11487 entries, 0 to 11486
Data columns (total 25 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   Date                                                                           11487 non-null  str    
 1   Service                                                                        11487 non-null  str    
 2   Departure station                                                              11487 non-null  str    
 3   Arrival station                                                                11487 non-null  str    
 4   Average journey time                                                           11487 non-null  float64
 5   Number of scheduled trains                                                     11487 non-null  float64
 6   Numb

## Cleaning and Conversion

Cette partie ne devrait pas normalement exister. Mais au vue du retard des autres membres de l'équipe, je me vois obliger de faire un petit truc pour pouvoir tester des modèles. Il s'agira donc de sélectionner les colonnes numériques, de les convertir puis de supprimer les lignes contenant une ou plusieurs valeurs manquantes au niveau de ces colonnes.

In [4]:
# Liste de quelques colonnes numériques
cols_numeriques = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Average delay of all trains at arrival"
]

# Conversion des colonnes en type numériques
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
# Reduction du nombre de colonnes
df = df[cols_numeriques]

# Taille du nouveau dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du nouveau dataset
print("\nAperçu du nouveau dataset:")
df.head()

Nombre d'échantillons: 11487
Nombre de caractéristiques: 4

Aperçu du nouveau dataset:


,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of all trains at arrival
0,870.0,5.0,289.0,6.511118
1,406.0,1.0,213.0,5.363539
2,226.0,0.0,21.0,2.938053
3,508.0,3.0,71.0,5.292211
4,472.0,4.0,224.0,4.882372


In [6]:
# Nombre de valeurs manquantes
df.isnull().sum()

Number of scheduled trains                0
Number of cancelled trains                0
Number of trains delayed at departure     0
Average delay of all trains at arrival    0
dtype: int64

In [7]:
# Suppression des valeurs manquantes
df = df.dropna()

In [8]:
# Taille du dataset après suppression des valeurs manquantes
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

Nombre d'échantillons: 11487
Nombre de caractéristiques: 4


## Choix des variables

In [9]:
# Définition de X (features) et y (target)
X = df[["Number of scheduled trains",
        "Number of cancelled trains",
        "Number of trains delayed at departure"]]
y = df["Average delay of all trains at arrival"]

# Division en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

## Entrainement et Evaluation

In [10]:
# Entrainement
model = LinearRegression()
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE\t:{rmse: .2f} minutes")
print(f"MAE\t:{mae: .2f} minutes")
print(f"R2\t:{r2: .4f}")

RMSE	: 10.72 minutes
MAE	: 3.06 minutes
R2	: 0.0043


## Sauvegarde du modèle

In [11]:
joblib.dump(model, "model.joblib")
print("Modèle sauvegardé avec succès")

Modèle sauvegardé avec succès
